# M5 Probabilistic Forecasting — Evaluacion (Fase 6)

**Objetivo:** comparativa honesta entre ARIMA clasico, BQML ARIMA_PLUS y LightGBM Cuantil sobre walk-forward CV (5 folds, Fase 5), mas analisis narrativo de casos dificiles pedido en `INSTRUCCIONES.md`.

**Tablas consumidas (todas ya construidas en BigQuery, ver `src/evaluation/`):**
- `cv_pinball_loss`, `cv_metrics_by_fold_quantile`, `cv_metrics_by_category`, `cv_metrics_overall` — `build_cv_metrics.py`
- `cv_metrics_by_product_category`, `cv_metrics_by_release_age`, `cv_metrics_by_event` — `build_case_analysis.py`
- `arima_metadata_cv` — convergencia de ARIMA por fold (Fase 5)

**Nota de alcance:** la comparacion "justa" (3 modelos) esta limitada a las 32 series de `arima_sample` (`in_arima_sample=TRUE`) porque ARIMA clasico nunca corrio sobre `lgbm_sample` completo (ver `phase-summaries/04-modelos.md`). BQML vs. LightGBM sobre las ~3,000 series de `lgbm_sample` se reporta aparte.

**Proyecto:** mle-m5-forecast | **Dataset:** m5_dataset


In [ ]:
# Setup
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT = 'mle-m5-forecast'
DATASET = 'm5_dataset'
client = bigquery.Client(project=PROJECT)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
MODEL_ORDER = ['arima', 'bqml', 'lgbm']
MODEL_LABELS = {'arima': 'ARIMA', 'bqml': 'BQML ARIMA_PLUS', 'lgbm': 'LightGBM Cuantil'}

def weighted_pivot(df, index_col, index_order=None):
    """Promedio ponderado por n_obs antes de pivotear (colapsa filas
    duplicadas de un mismo index_col con distinto valor de in_arima_sample)."""
    w = df.assign(weighted=df['avg_pinball_loss'] * df['n_obs'])
    agg = w.groupby([index_col, 'model'], as_index=False).agg(weighted_sum=('weighted', 'sum'), n_obs_sum=('n_obs', 'sum'))
    agg['avg_pinball_loss'] = agg['weighted_sum'] / agg['n_obs_sum']
    pivot = agg.pivot(index=index_col, columns='model', values='avg_pinball_loss')
    if index_order is not None:
        pivot = pivot.reindex([b for b in index_order if b in pivot.index])
    return pivot[[m for m in MODEL_ORDER if m in pivot.columns]]

def fair_and_full(df, index_col, index_order=None):
    """Split de dos alcances (mismo principio que Seccion 1): arima SOLO
    tiene predicciones para las 32 series de arima_sample -- comparar la
    fila 'arima' contra un bqml/lgbm promediado sobre lgbm_sample completo
    mezcla poblaciones distintas. Devuelve (pivot_justo_3_modelos,
    pivot_bqml_vs_lgbm_scope_completo)."""
    fair = weighted_pivot(df[df['in_arima_sample']], index_col, index_order)
    full = weighted_pivot(df[df['model'].isin(['bqml', 'lgbm'])], index_col, index_order)
    return fair, full

print(f'✓ Conectado a {PROJECT}.{DATASET}')

## 1. Comparativa general de modelos (walk-forward CV, 5 folds)

Pinball Loss promedio por percentil. Resultado ya documentado en `phase-summaries/05-walk-forward-cv.md` y `README.md` — aca se reproduce con codigo ejecutable y se agrega el grafico.

In [ ]:
query = f"""
SELECT model, quantile_name, quantile_level, in_arima_sample, n_obs, avg_pinball_loss
FROM `{PROJECT}.{DATASET}.cv_metrics_overall`
ORDER BY model, quantile_level
"""
df_overall = client.query(query).to_dataframe()

print('=== Comparacion justa -- 32 series de arima_sample, presentes en los 3 modelos ===')
fair = df_overall[df_overall['in_arima_sample']]
pivot_fair = fair.pivot(index='quantile_name', columns='model', values='avg_pinball_loss').reindex(
    ['p05', 'p25', 'p50', 'p75', 'p95'])[MODEL_ORDER]
display(pivot_fair)

print('\n=== BQML vs LightGBM -- scope completo (~3,000 series de lgbm_sample) ===')
df_overall['weighted'] = df_overall['avg_pinball_loss'] * df_overall['n_obs']
full_scope = df_overall.groupby(['model', 'quantile_name', 'quantile_level'], as_index=False).agg(
    weighted_sum=('weighted', 'sum'), n_obs_sum=('n_obs', 'sum'))
full_scope['avg_pinball_loss'] = full_scope['weighted_sum'] / full_scope['n_obs_sum']
pivot_full = full_scope[full_scope['model'].isin(['bqml', 'lgbm'])].pivot(
    index='quantile_name', columns='model', values='avg_pinball_loss').reindex(
    ['p05', 'p25', 'p50', 'p75', 'p95'])
display(pivot_full)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for model in MODEL_ORDER:
    sub = pivot_fair[model]
    ax.plot(sub.index, sub.values, marker='o', label=MODEL_LABELS[model], linewidth=2)
ax.set_title('Pinball Loss por percentil -- comparacion justa (32 series, 3 modelos)', fontsize=14)
ax.set_xlabel('Percentil')
ax.set_ylabel('Pinball Loss (promedio, 5 folds)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Pinball Loss por categoria real (FOODS / HOBBIES / HOUSEHOLD)

Fuente: `cv_metrics_by_product_category` (`src/evaluation/build_case_analysis.py`). Un modelo puede ser bueno en P50 global y malo en una categoria especifica -- ver `skill-m5-evaluation.md`.

In [ ]:
query = f"""
SELECT model, quantile_name, quantile_level, category, in_arima_sample, n_obs, avg_pinball_loss
FROM `{PROJECT}.{DATASET}.cv_metrics_by_product_category`
ORDER BY category, model, quantile_level
"""
df_cat = client.query(query).to_dataframe()

cat_order = ['FOODS', 'HOBBIES', 'HOUSEHOLD']
p50_cat = df_cat[df_cat['quantile_name'] == 'p50']
pivot_cat_fair, pivot_cat_full = fair_and_full(p50_cat, 'category', cat_order)

print('=== Comparacion justa (32 series arima_sample, 3 modelos) ===')
display(pivot_cat_fair)
print('\n=== BQML vs LightGBM, scope completo (~3,000 series lgbm_sample) ===')
display(pivot_cat_full)

In [ ]:
pivot_cat_full.plot(kind='bar', figsize=(10, 6), color=['#DD8452', '#55A868'])
plt.title('Pinball Loss P50 por categoria de producto (BQML vs LightGBM, scope completo)', fontsize=13)
plt.ylabel('Pinball Loss')
plt.xlabel('Categoria')
plt.xticks(rotation=0)
plt.legend(title='Modelo')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Casos dificiles

Tres segmentos pedidos en `INSTRUCCIONES.md` / `skill-m5-evaluation.md`: series con alta tasa de ceros, productos nuevos sin historia, semanas con eventos especiales.

### 3.1 Series con alta tasa de ceros

Fuente: `cv_metrics_by_category` (`categoria_zero_rate`, Fase 5). Buckets: `rapido` (<20% ceros), `medio` (20-50%), `lento` (50-80%), `muy_lento` (>=80%).

In [ ]:
query = f"""
SELECT model, quantile_name, quantile_level, categoria_zero_rate, in_arima_sample, n_obs, avg_pinball_loss
FROM `{PROJECT}.{DATASET}.cv_metrics_by_category`
WHERE quantile_name = 'p50'
ORDER BY categoria_zero_rate, model
"""
df_zero = client.query(query).to_dataframe()
zero_order = ['rapido', 'medio', 'lento', 'muy_lento']
pivot_zero_fair, pivot_zero_full = fair_and_full(df_zero, 'categoria_zero_rate', zero_order)

print('=== Comparacion justa (32 series arima_sample, 3 modelos) ===')
display(pivot_zero_fair)
print('\n=== BQML vs LightGBM, scope completo (~3,000 series lgbm_sample) ===')
display(pivot_zero_full)

pivot_zero_full.plot(kind='bar', figsize=(10, 6), color=['#DD8452', '#55A868'])
plt.title('Pinball Loss P50 por tasa de ceros (BQML vs LightGBM, scope completo)', fontsize=13)
plt.ylabel('Pinball Loss')
plt.xticks(rotation=0)
plt.legend(title='Modelo')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 3.2 Productos nuevos sin historia

Fuente: `cv_metrics_by_release_age` (`src/evaluation/build_case_analysis.py`). `release_date` = primera semana con precio registrado en `sell_prices` (definicion oficial M5). `nuevo_lt_90d` = prediccion dentro de los primeros 90 dias desde el release.

In [ ]:
query = f"""
SELECT model, quantile_name, quantile_level, release_age_bucket, in_arima_sample, n_obs, avg_pinball_loss
FROM `{PROJECT}.{DATASET}.cv_metrics_by_release_age`
WHERE quantile_name = 'p50'
ORDER BY release_age_bucket, model
"""
df_age = client.query(query).to_dataframe()
print('n_obs por bucket (scope completo bqml+lgbm, confirma si hay suficiente muestra de productos nuevos):')
display(df_age[df_age['model'].isin(['bqml', 'lgbm'])].groupby('release_age_bucket')['n_obs'].sum())

age_order = [b for b in ['nuevo_lt_90d', 'establecido', 'antes_de_release', 'sin_release_date'] if b in df_age['release_age_bucket'].unique()]
pivot_age_fair, pivot_age_full = fair_and_full(df_age, 'release_age_bucket', age_order)

print('\n=== Comparacion justa (32 series arima_sample, 3 modelos) ===')
display(pivot_age_fair)
print('\n=== BQML vs LightGBM, scope completo (~3,000 series lgbm_sample) ===')
display(pivot_age_full)

pivot_age_full.plot(kind='bar', figsize=(10, 6), color=['#DD8452', '#55A868'])
plt.title('Pinball Loss P50 por antiguedad de release (BQML vs LightGBM, scope completo)', fontsize=13)
plt.ylabel('Pinball Loss')
plt.xticks(rotation=20)
plt.legend(title='Modelo')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 3.3 Semanas con eventos especiales

Fuente: `cv_metrics_by_event`. `navidad` = 25-dic (Walmart cierra, caso extremo del EDA); `evento` = cualquier otro evento de calendario (`event_name_1`/`event_name_2`); `sin_evento` = resto de los dias.

In [ ]:
query = f"""
SELECT model, quantile_name, quantile_level, event_bucket, in_arima_sample, n_obs, avg_pinball_loss
FROM `{PROJECT}.{DATASET}.cv_metrics_by_event`
WHERE quantile_name = 'p50'
ORDER BY event_bucket, model
"""
df_event = client.query(query).to_dataframe()
print('n_obs por bucket (scope completo bqml+lgbm):')
display(df_event[df_event['model'].isin(['bqml', 'lgbm'])].groupby('event_bucket')['n_obs'].sum())

event_order = [b for b in ['navidad', 'evento', 'sin_evento'] if b in df_event['event_bucket'].unique()]
pivot_event_fair, pivot_event_full = fair_and_full(df_event, 'event_bucket', event_order)

print('\n=== Comparacion justa (32 series arima_sample, 3 modelos) ===')
display(pivot_event_fair)
print('\n=== BQML vs LightGBM, scope completo (~3,000 series lgbm_sample) ===')
display(pivot_event_full)

pivot_event_full.plot(kind='bar', figsize=(10, 6), color=['#DD8452', '#55A868'])
plt.title('Pinball Loss P50 por tipo de dia (BQML vs LightGBM, scope completo)', fontsize=13)
plt.ylabel('Pinball Loss')
plt.xticks(rotation=0)
plt.legend(title='Modelo')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Convergencia de ARIMA por fold

Insumo adicional para "productos nuevos sin historia": la convergencia de ARIMA clasico cae fuerte en los folds tempranos (ver `phase-summaries/05-walk-forward-cv.md`), consistente con series que aun no acumulan suficiente historia en 2011-2012.

In [ ]:
query = f"""
SELECT fold_id, convergio, COUNT(*) AS n_series
FROM `{PROJECT}.{DATASET}.arima_metadata_cv`
GROUP BY fold_id, convergio
ORDER BY fold_id, convergio
"""
df_conv = client.query(query).to_dataframe()
pivot_conv = df_conv.pivot(index='fold_id', columns='convergio', values='n_series').fillna(0)
pivot_conv['pct_converged'] = 100 * pivot_conv.get(True, 0) / pivot_conv.sum(axis=1)
display(pivot_conv)

plt.figure(figsize=(9, 5))
plt.plot(pivot_conv.index, pivot_conv['pct_converged'], marker='o', linewidth=2, color='darkred')
plt.title('% de series ARIMA que convergen, por fold', fontsize=14)
plt.xlabel('Fold (1 = mas antiguo, 5 = mas reciente)')
plt.ylabel('% convergencia')
plt.ylim(0, 100)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Conclusiones

*(Completar despues de correr las celdas anteriores en la Workstation -- estructura de la narrativa a seguir, ver `skill-m5-evaluation.md`):*

- **Comparativa general:** LightGBM Cuantil gana en los 5 percentiles, en ambos alcances de comparacion (ver Seccion 1). BQML supera a ARIMA clasico en el cuerpo de la distribucion (P25-P75) pero pierde en las colas (P05/P95) -- coherente con que `ARIMA_PLUS` asume normalidad para sus intervalos.
- **Por categoria de producto:** completar tras revisar Seccion 2 -- ¿hay alguna categoria donde el ranking de modelos cambia?
- **Series con alta tasa de ceros:** completar tras revisar Seccion 3.1 -- hipotesis a validar: la ventaja de LightGBM deberia ampliarse en `lento`/`muy_lento` (demanda intermitente, 78.4% de las series segun el EDA).
- **Productos nuevos:** completar tras revisar Seccion 3.2 y 4 -- la caida de convergencia de ARIMA en folds tempranos (Seccion 4) sugiere que el segmento `nuevo_lt_90d` deberia mostrar la brecha mas grande a favor de LightGBM/BQML.
- **Eventos especiales:** completar tras revisar Seccion 3.3 -- ¿el bucket `navidad` muestra error mucho mayor en todos los modelos (cierre de tienda, caso extremo) comparado con `evento`/`sin_evento`?
- **Narrativa final para el portfolio:** ARIMA establece el piso estadistico, BQML escala eso a produccion pero mantiene las limitaciones de distribucion normal, LightGBM Cuantil supera ambos y agrega incertidumbre real -- confirmar que cada paso de esta narrativa esta respaldado por los numeros de arriba antes de escribirla en el README/presentacion final (Fase 10).